In [1]:
import os
import glob
import pandas as pd
import numpy as np
import krippendorff
from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
from scipy.stats import wilcoxon
from sklearn.metrics import f1_score
import itertools

# Interrater Agreement

In [2]:
#define file paths and categories

file_paths_human = [
    "../1_preprocessing/annotation/headlines_dataset_15sample_annotation_alvaro.xlsx", 
    "../1_preprocessing/annotation/headlines_dataset_15sample_annotiert_Felix.xlsx", 
    "../1_preprocessing/annotation/headlines_dataset_15sample_annotiert_Tai.xlsx"
]

file_paths_LMM = [
    "../1_preprocessing/annotation/headlines_dataset_15sample_annotation_ChatGPT.xlsx", 
    "../1_preprocessing/annotation/headlines_dataset_15sample_annotation_ChatGPTPlus.xlsx", 
    "../1_preprocessing/annotation/headlines_dataset_15sample_annotation_Gemini.xlsx",
    "../1_preprocessing/annotation/headlines_dataset_15sample_annotation-Claude.xlsx"
]

categories = [
    "Economic",
    "Capacity and resources",
    "Morality",
    "Fairness and equality",
    "Legality, constitutionality and jurisprudence",
    "Policy prescription and evaluation",
    "Crime and punishment",
    "Security and defense",
    "Health and safety",
    "Quality of life",
    "Cultural identity",
    "Public opinion",
    "Political",
    "External regulation and reputation"
]

## Load Agreement Files

In [3]:
# load files with their categorial annotation (0, 1, 2)
dfs_human = [pd.read_excel(path) for path in file_paths_human]
dfs_LLM = [pd.read_excel(path) for path in file_paths_LMM]

In [4]:
# turn human categorial data into binary data
dfs_human_binary = []

for df in dfs_human:
    df_copy = df.copy()
    df_copy[categories] = (df_copy[categories] > 0).astype(int)
    dfs_human_binary.append(df_copy)

In [5]:
# turn LLM categorial data into binary data
dfs_LLM_binary = []

for df in dfs_LLM:
    df_copy = df.copy()
    df_copy[categories] = (df_copy[categories] > 0).astype(int)
    dfs_LLM_binary.append(df_copy)

In [ ]:
# checking unique labels for LLM -> Quality of Life only has '0' as labels!
for i, df in enumerate(dfs_LLM):
    print(f"Annotator {i}:")
    for cat in categories:
        print(cat, df[cat].unique())

Annotator 0:
Economic [0 1 2]
Capacity and resources [0 2 1]
Morality [0 2 1]
Fairness and equality [0 2 1]
Legality, constitutionality and jurisprudence [2 0 1]
Policy prescription and evaluation [0 1 2]
Crime and punishment [0 1 2]
Security and defense [0 1 2]
Health and safety [0 1 2]
Quality of life [0]
Cultural identity [0 2 1]
Public opinion [0 1 2]
Political [1 2 0]
External regulation and reputation [0 2 1]
Annotator 1:
Economic [0 1 2]
Capacity and resources [0 2 1]
Morality [0 2 1]
Fairness and equality [0 2 1]
Legality, constitutionality and jurisprudence [2 0 1]
Policy prescription and evaluation [0 2 1]
Crime and punishment [0 2 1]
Security and defense [0 1 2]
Health and safety [0 1 2]
Quality of life [0 1 2]
Cultural identity [0 1 2]
Public opinion [1 0 2]
Political [1 2 0]
External regulation and reputation [0 1 2]
Annotator 2:
Economic [0 1 2]
Capacity and resources [0 2 1]
Morality [0 2 1]
Fairness and equality [0 2 1]
Legality, constitutionality and jurisprudence [2 0

## calcuate metrics: human

In [7]:
# human
results = []

# calculate metrics per category
for cat in categories:
    # Construct a matrix of shape (n_annotators, n_samples) for Krippendorff's Alpha
    # Example matrix: Row 0 = Annotator 1 ratings, Row 1 = Annotator 2, etc.
    alpha_matrix = np.array([df[cat].values for df in dfs_human])
    
    # Krippendorff's Alpha (Ordinal accounts for distance: 0 vs 2 is worse than 0 vs 1)
    k_alpha_ordinal = krippendorff.alpha(reliability_data=alpha_matrix, level_of_measurement='ordinal')
    k_alpha_nominal = krippendorff.alpha(reliability_data=alpha_matrix, level_of_measurement='nominal')
    
    # Construct Fleiss' Kappa table (shape: n_samples x n_categories_count)
    # Pivot so each row contains counts of [0s, 1s, 2s] assigned to that item
    sample_ratings = np.column_stack([df[cat].values for df in dfs_human])
    fleiss_data, _ = aggregate_raters(sample_ratings)
    f_kappa = fleiss_kappa(fleiss_data)
    
    results.append({
        "Category": cat,
        "Krippendorff_Alpha_Ordinal": round(k_alpha_ordinal, 4),
        "Krippendorff_Alpha_Nominal": round(k_alpha_nominal, 4),
        "Fleiss_Kappa": round(f_kappa, 4)
    })

# save and view summary dataframe
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))
print(np.mean(df_results['Krippendorff_Alpha_Nominal']))
print(np.mean(df_results['Krippendorff_Alpha_Ordinal']))

                                     Category  Krippendorff_Alpha_Ordinal  Krippendorff_Alpha_Nominal  Fleiss_Kappa
                                     Economic                      0.5485                      0.4254        0.4121
                       Capacity and resources                      0.4222                      0.2875        0.2907
                                     Morality                      0.3438                      0.2299        0.2376
                        Fairness and equality                      0.2660                      0.1974        0.2184
Legality, constitutionality and jurisprudence                      0.1276                      0.0739        0.1214
           Policy prescription and evaluation                      0.0777                      0.0684        0.0996
                         Crime and punishment                      0.5161                      0.3975        0.3889
                         Security and defense                      0.498

In [8]:
# human
results = []

# calculate metrics per category
for cat in categories:
    # Construct a matrix of shape (n_annotators, n_samples) for Krippendorff's Alpha
    # Example matrix: Row 0 = Annotator 1 ratings, Row 1 = Annotator 2, etc.
    alpha_matrix = np.array([df[cat].values for df in dfs_human_binary])
    
    # Krippendorff's Alpha (Ordinal accounts for distance: 0 vs 2 is worse than 0 vs 1)
    #k_alpha_ordinal = krippendorff.alpha(reliability_data=alpha_matrix, level_of_measurement='ordinal')
    k_alpha_nominal = krippendorff.alpha(reliability_data=alpha_matrix, level_of_measurement='nominal')
    
    # Construct Fleiss' Kappa table (shape: n_samples x n_categories_count)
    # Pivot so each row contains counts of [0s, 1s, 2s] assigned to that item
    sample_ratings = np.column_stack([df[cat].values for df in dfs_human])
    fleiss_data, _ = aggregate_raters(sample_ratings)
    f_kappa = fleiss_kappa(fleiss_data)
    
    results.append({
        "Category": cat,
        #"Krippendorff_Alpha_Ordinal": round(k_alpha_ordinal, 4),
        "Krippendorff_Alpha_Nominal": round(k_alpha_nominal, 4),
        "Fleiss_Kappa": round(f_kappa, 4)
    })

# save and view summary dataframe
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))
np.mean(df_results['Krippendorff_Alpha_Nominal'])

                                     Category  Krippendorff_Alpha_Nominal  Fleiss_Kappa
                                     Economic                      0.5349        0.4121
                       Capacity and resources                      0.4066        0.2907
                                     Morality                      0.3112        0.2376
                        Fairness and equality                      0.2657        0.2184
Legality, constitutionality and jurisprudence                      0.1249        0.1214
           Policy prescription and evaluation                      0.0762        0.0996
                         Crime and punishment                      0.4993        0.3889
                         Security and defense                      0.4782        0.3182
                            Health and safety                      0.6160        0.4919
                              Quality of life                      0.0139        0.0806
                            Cult

np.float64(0.31437857142857145)

## calcuate metrics: LLM

In [9]:
# LLM
results = []

# calculate metrics per category
for cat in categories:
    # Construct a matrix of shape (n_annotators, n_samples) for Krippendorff's Alpha
    # Example matrix: Row 0 = Annotator 1 ratings, Row 1 = Annotator 2, etc.
    alpha_matrix = np.array([df[cat].values for df in dfs_LLM])
    
    # Krippendorff's Alpha (Ordinal accounts for distance: 0 vs 2 is worse than 0 vs 1)
    k_alpha_ordinal = krippendorff.alpha(reliability_data=alpha_matrix, level_of_measurement='ordinal')
    k_alpha_nominal = krippendorff.alpha(reliability_data=alpha_matrix, level_of_measurement='nominal')
    
    # Construct Fleiss' Kappa table (shape: n_samples x n_categories_count)
    # Pivot so each row contains counts of [0s, 1s, 2s] assigned to that item
    sample_ratings = np.column_stack([df[cat].values for df in dfs_LLM])
    fleiss_data, _ = aggregate_raters(sample_ratings)
    f_kappa = fleiss_kappa(fleiss_data)
    
    results.append({
        "Category": cat,
        "Krippendorff_Alpha_Ordinal": round(k_alpha_ordinal, 4),
        "Krippendorff_Alpha_Nominal": round(k_alpha_nominal, 4),
        "Fleiss_Kappa": round(f_kappa, 4)
    })

# save and view summary dataframe
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))
print(np.mean(df_results['Krippendorff_Alpha_Nominal']))
print(np.mean(df_results['Krippendorff_Alpha_Ordinal']))

                                     Category  Krippendorff_Alpha_Ordinal  Krippendorff_Alpha_Nominal  Fleiss_Kappa
                                     Economic                      0.7806                      0.6645        0.6641
                       Capacity and resources                      0.5951                      0.4745        0.4739
                                     Morality                      0.7341                      0.6366        0.6362
                        Fairness and equality                      0.8281                      0.7453        0.7450
Legality, constitutionality and jurisprudence                      0.6092                      0.4902        0.4896
           Policy prescription and evaluation                      0.4543                      0.3473        0.3466
                         Crime and punishment                      0.8096                      0.7154        0.7150
                         Security and defense                      0.720

In [10]:
# LLM
results = []

# calculate metrics per category
for cat in categories:
    # Construct a matrix of shape (n_annotators, n_samples) for Krippendorff's Alpha
    # Example matrix: Row 0 = Annotator 1 ratings, Row 1 = Annotator 2, etc.
    alpha_matrix = np.array([df[cat].values for df in dfs_LLM_binary])
    
    # Krippendorff's Alpha (Ordinal accounts for distance: 0 vs 2 is worse than 0 vs 1)
    #k_alpha_ordinal = krippendorff.alpha(reliability_data=alpha_matrix, level_of_measurement='ordinal')
    k_alpha_nominal = krippendorff.alpha(reliability_data=alpha_matrix, level_of_measurement='nominal')
    
    # Construct Fleiss' Kappa table (shape: n_samples x n_categories_count)
    # Pivot so each row contains counts of [0s, 1s, 2s] assigned to that item
    sample_ratings = np.column_stack([df[cat].values for df in dfs_LLM])
    fleiss_data, _ = aggregate_raters(sample_ratings)
    f_kappa = fleiss_kappa(fleiss_data)
    
    results.append({
        "Category": cat,
        #"Krippendorff_Alpha_Ordinal": round(k_alpha_ordinal, 4),
        "Krippendorff_Alpha_Nominal": round(k_alpha_nominal, 4),
        "Fleiss_Kappa": round(f_kappa, 4)
    })

# save and view summary dataframe
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))
print(np.mean(df_results['Krippendorff_Alpha_Nominal']))

                                     Category  Krippendorff_Alpha_Nominal  Fleiss_Kappa
                                     Economic                      0.7585        0.6641
                       Capacity and resources                      0.5769        0.4739
                                     Morality                      0.7155        0.6362
                        Fairness and equality                      0.8202        0.7450
Legality, constitutionality and jurisprudence                      0.5938        0.4896
           Policy prescription and evaluation                      0.4361        0.3466
                         Crime and punishment                      0.7925        0.7150
                         Security and defense                      0.6860        0.5797
                            Health and safety                      0.6835        0.6083
                              Quality of life                      0.0054       -0.0091
                            Cult

# Create full Ground Truth Excel

## binary ground truth (no main frame)

In [11]:
def build_labels_full(base_dir):

    # 14 labels formatted exactly as they appear in data
    LABELS = [
        "Economic",
        "Capacity_and_resources",
        "Morality",
        "Fairness_and_equality",
        "Legality_Constitutionality_and_jurisprudence",
        "Policy_prescription_and_evaluation",
        "Crime_and_punishment",
        "Security_and_defense",
        "Health_and_safety",
        "Quality_of_life",
        "Cultural_identity",
        "Public_opinion",
        "Political",
        "External_regulation_and_reputation"
    ]
    
    # Target files (add '.txt' if your files have extensions)
    target_files = ['train-labels-subtask-2.txt', 'dev-labels-subtask-2.txt']
    
    data = []
    
    # iterate through every item in the base directory
    for lang_code in os.listdir(base_dir):
        folder_path = os.path.join(base_dir, lang_code)
        
        # ensure we are only looking at directories (the language folders)
        if os.path.isdir(folder_path):
            
            # check for both train and dev files in the language folder
            for file_name in target_files:
                file_path = os.path.join(folder_path, file_name)
                
                # check if the file exists (and also check for .txt fallback just in case)
                if not os.path.exists(file_path) and os.path.exists(file_path + ".txt"):
                    file_path += ".txt"
                    
                if os.path.exists(file_path):
                    with open(file_path, 'r', encoding='utf-8') as file:
                        for line in file:
                            line = line.strip()
                            if not line:
                                continue # skip empty lines
                            
                            # split by tab to separate article_id and the labels
                            parts = line.split('\t')
                            if len(parts) != 2:
                                continue # skip malformed lines
                                
                            article_id, labels_str = parts
                            
                            # initialize the row dictionary
                            row = {
                                'Filename': f'article{article_id}.txt',
                                'Language': lang_code
                            }
                            
                            # set default value of 0 for all 14 labels
                            for label in LABELS:
                                row[label] = 0
                                
                            # split the comma-separated labels and set them to 1
                            current_labels = labels_str.split(',')
                            for label in current_labels:
                                label = label.strip() # Remove any accidental whitespace
                                if label in row:
                                    row[label] = 1
                                    
                            data.append(row)
                            
    # convert the list of dictionaries into a Pandas DataFrame
    df = pd.DataFrame(data)
    
    return df

In [12]:
# Execution

base_directory = '../0_data' 
df_true_full = build_labels_full(base_directory)

df_true_full.columns = df_true_full.columns.str.replace('_', ' ')
df_true_full.rename(columns={"Legality Constitutionality and jurisprudence": "Legality, constitutionality and jurisprudence"}, inplace=True)

output_filename = 'ground_truth_full.xlsx'
df_true_full.to_excel(output_filename, index=False)

## categorial ground truth
first label as main frame approximation

In [13]:
def build_labels_partial(base_dir):

    LABELS = [
        "Economic",
        "Capacity_and_resources",
        "Morality",
        "Fairness_and_equality",
        "Legality_Constitutionality_and_jurisprudence",
        "Policy_prescription_and_evaluation",
        "Crime_and_punishment",
        "Security_and_defense",
        "Health_and_safety",
        "Quality_of_life",
        "Cultural_identity",
        "Public_opinion",
        "Political",
        "External_regulation_and_reputation"
    ]
    
    target_files = ['train-labels-subtask-2.txt', 'dev-labels-subtask-2.txt']
    
    data = []
    
    for lang_code in os.listdir(base_dir):
        folder_path = os.path.join(base_dir, lang_code)
        
        if os.path.isdir(folder_path):
            
            for file_name in target_files:
                file_path = os.path.join(folder_path, file_name)
                
                if not os.path.exists(file_path) and os.path.exists(file_path + ".txt"):
                    file_path += ".txt"
                    
                if os.path.exists(file_path):
                    with open(file_path, 'r', encoding='utf-8') as file:
                        for line in file:
                            line = line.strip()
                            if not line:
                                continue 
                            
                            parts = line.split('\t')
                            if len(parts) != 2:
                                continue
                                
                            article_id, labels_str = parts
                            
                            row = {
                                'Filename': f'article{article_id}.txt',
                                'Language': lang_code
                            }
                            
                            for label in LABELS:
                                row[label] = 0
                                
                            current_labels = [l.strip() for l in labels_str.split(',') if l.strip()]
                            
                            for idx, label in enumerate(current_labels):
                                if label in row:
                                    # NEW: 2 for the first label (idx == 0), 1 for all following labels
                                    row[label] = 2 if idx == 0 else 1
                                    
                            data.append(row)
                            
    df = pd.DataFrame(data)
    
    return df

In [14]:
# Execution
base_directory = '../0_data' 

df_true_partial = build_labels_partial(base_directory)

df_true_partial.columns = df_true_partial.columns.str.replace('_', ' ')
df_true_partial.rename(columns={"Legality Constitutionality and jurisprudence": "Legality, constitutionality and jurisprudence"}, inplace=True)

output_filename = 'ground_truth_partial.xlsx'
df_true_partial.to_excel(output_filename, index=False)

# find best annotator: human or LLM?

In [ ]:
GROUND_TRUTH_FULL = 'ground_truth_full.xlsx'
GROUND_TRUTH_PARTIAL = 'ground_truth_partial.xlsx' # only mainframe is 1

ANNOTATION_DIR = '../1_preprocessing/annotation/'

# Metrics with simplified Ground Truth

In [19]:
# load Ground Truth
gt_df = pd.read_excel(GROUND_TRUTH_FULL)

# ensure article_id is treated as string/object to avoid type mismatch on merge
gt_df['Filename'] = gt_df['Filename'].astype(str)

# normalize Ground Truth binary values (ensure boolean 1 or 0)
gt_df[categories] = (gt_df[categories] > 0).astype(int)

# process each annotator file in the 'annotation' folder
annotator_files = glob.glob(os.path.join(ANNOTATION_DIR, '*.xlsx')) + \
                  glob.glob(os.path.join(ANNOTATION_DIR, '*.csv'))

In [20]:
results = []

for file_path in annotator_files:
    annotator_name = os.path.basename(file_path).split('.')[0]
    
    # Read file
    if file_path.endswith('.xlsx'):
        ann_df = pd.read_excel(file_path)
        
    ann_df['Filename'] = ann_df['Filename'].astype(str)
    
    # check that required columns exist
    missing_cols = [col for col in categories if col not in ann_df.columns]
    if missing_cols:
        print(f"Skipping {annotator_name}: missing label columns {missing_cols}")
        continue
        
    # CRITICAL: CONVERT 1+2 to 1, and 0 to 0
    ann_df[categories] = (ann_df[categories] > 0).astype(int)
    
    # align annotator data with Ground Truth on article_id (inner join picks ONLY annotated sample)
    merged = pd.merge(
        gt_df[['Filename'] + categories], 
        ann_df[['Filename'] + categories], 
        on='Filename', 
        suffixes=('_gt', '_ann')
    )
    
    # extract aligned matrices
    y_true = merged[[f"{c}_gt" for c in categories]].values
    y_pred = merged[[f"{c}_ann" for c in categories]].values
    
    # calculate Metrics
    micro_f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    precision = precision_score(y_true, y_pred, average='micro', zero_division=0)
    recall = recall_score(y_true, y_pred, average='micro', zero_division=0)
    
    results.append({
        'Annotator': annotator_name,
        'Annotated Sample Size': len(merged),
        'Micro F1 (Overall Agreement)': round(micro_f1, 4),
        'Macro F1 (Label Average)': round(macro_f1, 4),
        'Precision': round(precision, 4),
        'Recall': round(recall, 4)
    })

In [21]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='Micro F1 (Overall Agreement)', ascending=False).reset_index(drop=True)
print("\nRELIABILITY RANKING")
print(results_df.to_string(index=False))


RELIABILITY RANKING
                                        Annotator  Annotated Sample Size  Micro F1 (Overall Agreement)  Macro F1 (Label Average)  Precision  Recall
    headlines_dataset_15sample_annotation_ChatGPT                    210                        0.4635                    0.3929     0.7122  0.3435
     headlines_dataset_15sample_annotation_Gemini                    210                        0.4630                    0.4015     0.6859  0.3494
     headlines_dataset_15sample_annotation-Claude                    210                        0.4575                    0.3979     0.7487  0.3294
headlines_dataset_15sample_annotation_ChatGPTPlus                    210                        0.4517                    0.4138     0.6308  0.3518
         headlines_dataset_15sample_annotiert_Tai                    210                        0.4475                    0.3919     0.5059  0.4012
       headlines_dataset_15sample_annotiert_Felix                    210                   

# Metrics with full approximated Ground Truth

In [ ]:
gt_df = pd.read_excel(GROUND_TRUTH_PARTIAL)

gt_df['Filename'] = gt_df['Filename'].astype(str)

# NEW: Target ONLY the '2's. If a cell is 2, it becomes 1 (Positive). If it is 1 or 0, it becomes 0 (Negative).
gt_df[categories] = (gt_df[categories] == 2).astype(int)


annotator_files = glob.glob(os.path.join(ANNOTATION_DIR, '*.xlsx')) + \
                  glob.glob(os.path.join(ANNOTATION_DIR, '*.csv'))

In [24]:
results = []

for file_path in annotator_files:
    annotator_name = os.path.basename(file_path).split('.')[0]
    
    if file_path.endswith('.xlsx'):
        ann_df = pd.read_excel(file_path)
        
    ann_df['Filename'] = ann_df['Filename'].astype(str)
    
    missing_cols = [col for col in categories if col not in ann_df.columns]
    if missing_cols:
        print(f"Skipping {annotator_name}: missing label columns {missing_cols}")
        continue
        
    # NEW: target ONLY the '2's
    ann_df[categories] = (ann_df[categories] == 2).astype(int)
    
    merged = pd.merge(
        gt_df[['Filename'] + categories], 
        ann_df[['Filename'] + categories], 
        on='Filename', 
        suffixes=('_gt', '_ann')
    )
    
    if merged.empty:
        print(f"Warning: No matching Filename found for {annotator_name}.")
        continue

    y_true = merged[[f"{c}_gt" for c in categories]].values
    y_pred = merged[[f"{c}_ann" for c in categories]].values
    
    # metrics
    mean_f1  = f1_score(y_true, y_pred, average='samples', zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    
    precision = precision_score(y_true, y_pred, average='micro', zero_division=0)
    recall = recall_score(y_true, y_pred, average='micro', zero_division=0)

    results.append({
        'Annotator': annotator_name,
        'Annotated Sample Size': len(merged),
        'Mean F1 (Instance Average)': round(mean_f1, 4),
        'Micro F1 (Overall Agreement)': round(micro_f1, 4),
        'Macro F1 (Label Average)': round(macro_f1, 4),
        'Precision': round(precision, 4),
        'Recall': round(recall, 4)
    })

In [25]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='Micro F1 (Overall Agreement)', ascending=False).reset_index(drop=True)
print("\nRELIABILITY RANKING (PRIMARY FRAMES ONLY)")
print(results_df.to_string(index=False))


RELIABILITY RANKING (PRIMARY FRAMES ONLY)
                                        Annotator  Annotated Sample Size  Mean F1 (Instance Average)  Micro F1 (Overall Agreement)  Macro F1 (Label Average)  Precision  Recall
    headlines_dataset_15sample_annotation_ChatGPT                    210                      0.2762                        0.2843                    0.2278     0.2929  0.2762
     headlines_dataset_15sample_annotation_Gemini                    210                      0.2619                        0.2619                    0.2124     0.2619  0.2619
headlines_dataset_15sample_annotation_ChatGPTPlus                    210                      0.2524                        0.2524                    0.2133     0.2524  0.2524
     headlines_dataset_15sample_annotation-Claude                    210                      0.2381                        0.2451                    0.1967     0.2525  0.2381
       headlines_dataset_15sample_annotiert_Felix                    210     